In [ ]:
import sys,os
hsaDir = os.path.abspath("../../")
srcDir = os.path.join(hsaDir,"src")
sys.path.insert(0,srcDir)

In [ ]:
from heliostat_surface_analysis.loader.scaling_loader import ScalingLoader
from heliostat_surface_analysis.loader.default_loader import DefaultLoader
from heliostat_surface_analysis.preprocessor.basic_preprocessor import BasicPreprocessor
from heliostat_surface_analysis.preprocessor.no_preprocessor import NoPreprocessor
from heliostat_surface_analysis.mesher.delaunay_mesher import DelaunayMesher
from heliostat_surface_analysis.mesher.bspline_mesher import BSplineMesher
from heliostat_surface_analysis.analyser.fit_surface_analyser import FitSurfaceAnalyser
from heliostat_surface_analysis.analyser.fixed_focal_length_analyser import FixedFocalLengthAnalyser
from heliostat_surface_analysis.aggregator.aggregator import WeightedRMSAggregator
from heliostat_surface_analysis.analysis_manager import AnalysisManager
from heliostat_surface_analysis.plotting.plotting import plot_2d_surface,set_plot_font_size

In [ ]:
import pandas as pd
import progressbar
import matplotlib.pyplot as plt
import numpy as np
import random

In [ ]:
%matplotlib inline

In [ ]:
# for progressbar
widgets = [' [',
         progressbar.Timer(format= 'elapsed time: %(elapsed)s'),
         '] ',
           progressbar.Bar('*'),' (',
           progressbar.ETA(), ') ',
          ]

### Setup analysis components

In [ ]:
preprocessor = BasicPreprocessor()
meshers = {}
# NOTE: setting the max_aspect_ratio is important
meshers["Delaunay"] = DelaunayMesher({'max_aspect_ratio':10})
meshers["B-spline"] = BSplineMesher({'polynomial_degree':3,'dx':0.01,'dy':0.01})
analysers = {}
analysers["FixedFocalLength"] = None # placeholder for ordering, will be filled in below
analysers["FitSurface (paraboloid)"] = FitSurfaceAnalyser({"surface":"circular_paraboloid"})
analysers["FitSurface (deformed paraboloid)"] = FitSurfaceAnalyser({"surface":"anu_polynomial"})
aggregator = WeightedRMSAggregator()

In [ ]:
# currently only "PEG Field2 scans" and "circular paraboid validation" implemented, others can be added below
scansCase = "PEG Field2 scans"

In [ ]:
if scansCase=="PEG Field2 scans":
    loader = ScalingLoader({"units":"mm"})
    scansDir = os.path.join(hsaDir,"examples","command_line","circular_paraboloid_peg","scans")
    designFocalLengthsFile = os.path.join(hsaDir,"examples","command_line","circular_paraboloid_peg","design_focal_lengths.txt")
    analysers["FixedFocalLength"] = FixedFocalLengthAnalyser({"focal_length_file":designFocalLengthsFile})
elif scansCase=="circular paraboid validation":
    loader = DefaultLoader()
    scansDir = os.path.join(hsaDir,"examples","command_line","ideal_circular_paraboloid_random","scans")
    designFocalLengthsFile = os.path.join(hsaDir,"examples","command_line","ideal_circular_paraboloid_random","design_focal_lengths.txt")
    analysers["FixedFocalLength"] = FixedFocalLengthAnalyser({"focal_length":29.})
else:
    raise RuntimeError("not recognised")

In [ ]:
scanFiles = sorted(os.listdir(scansDir))

### Compare meshers

In [ ]:
analyser = analysers["FitSurface (paraboloid)"]
scanNames = []
focalLengths = []
slopeErrors = {}
for mesherName,_ in meshers.items():
    slopeErrors[mesherName] = []
bar = progressbar.ProgressBar(maxval=len(scanFiles)).start()
for i,scanFile in enumerate(scanFiles):
    scanPath = os.path.join(scansDir,scanFile)
    for mesherName,mesher in meshers.items():
        result = AnalysisManager.analyse_scan(scanPath,loader,preprocessor,mesher,analyser,aggregator,output_dir=None,plot=False,write_stls=False)
        slopeErrors[mesherName].append(result["aggregate"])
        focalLength = 1/4./result["coeff: a(x^2 + y^2)"]
        scanName = result["name"]
    # these are the same irrespective of what mesher is used
    scanNames.append(scanName)
    focalLengths.append(focalLength)
    bar.update(i+1)
bar.finish()

In [ ]:
dataDict = {}
dataDict["scan"] = scanNames
dataDict["f"] = focalLengths
for mesherName,mesher in meshers.items():
    dataDict[f"sigma ({mesherName})"] = slopeErrors[mesherName]
df = pd.DataFrame(dataDict)
df.set_index("scan",inplace=True)
df.head(n=150)

In [ ]:
df.loc[df['sigma (FixedFocalLength)'].idxmax()]

In [ ]:
df.loc[df['sigma (FixedFocalLength)'].idxmin()]

### Compare analysers

In [ ]:
mesher = meshers["Delaunay"]
scanNames = []
slopeErrors = {}
focalLengths = {}
for analyserName,_ in analysers.items():
    slopeErrors[analyserName] = []
    focalLengths[analyserName] = []
bar = progressbar.ProgressBar(maxval=len(scanFiles)).start()
for i,scanFile in enumerate(scanFiles):
    scanPath = os.path.join(scansDir,scanFile)
    for analyserName,analyser in analysers.items():
        result = AnalysisManager.analyse_scan(scanPath,loader,preprocessor,mesher,analyser,aggregator,output_dir=None,plot=False,write_stls=False)
        slopeErrors[analyserName].append(result["aggregate"])
        scanName = result["name"]
        if analyserName=="FitSurface (paraboloid)":
            f = 1/4./result["coeff: a(x^2 + y^2)"]
        elif analyserName=="FitSurface (deformed paraboloid)":
            # NOTE: this the geometric average of fx and fy, as a single 'focal length' doesn't exist for the deformed paraboloid
            f = 1/4./np.sqrt(result["coeff: cx^2"]**2 + result["coeff: gy^2"]**2)
        else:
            f = analyser.focal_length_model.get(scanName)
        focalLengths[analyserName].append(f)
    # these are the same irrespective of what analyser is used
    scanNames.append(scanName)
    bar.update(i+1)
bar.finish()

In [ ]:
dataDict = {}
dataDict["scan"] = scanNames
for analyserName,_ in analysers.items():
    dataDict[f"f ({analyserName})"] = focalLengths[analyserName]
    dataDict[f"sigma ({analyserName})"] = slopeErrors[analyserName]
df = pd.DataFrame(dataDict)
df["eps(f)"] = (df["f (FixedFocalLength)"]-df["f (FitSurface (paraboloid))"])/df["f (FitSurface (paraboloid))"]
df.set_index("scan",inplace=True)
with pd.ExcelWriter('facets_analysis.ods', engine="odf") as doc:
    df.to_excel(doc, sheet_name="Sheet1")
df.head(n=150)

In [ ]:
for analyserName,_ in analysers.items():
    print("%s, mean slope error = %f mrad"%(analyserName,df["sigma (%s)"%analyserName].mean()*1e3))

### plotting of slope error distribution for the scanned facet population

In [ ]:
f,ax = plt.subplots(3,1)
f.set_size_inches(14,18)
for i,analyserName in enumerate(analysers.keys()):
    # ax[i].set_title(analyserName)
    ax[i].hist(df["sigma (%s)"%analyserName]*1e3,bins=20,rwidth=0.85)
    min_ylim, max_ylim = ax[i].get_ylim()
    mean = df["sigma (%s)"%analyserName].mean()*1e3
    ax[i].axvline(mean, color='red', linestyle='dashed', linewidth=3)
    ax[i].text(mean*1.1, max_ylim*0.9, '$\mu$ = {:.2f} mrad'.format(mean),color='red',size=16)
    ax[i].grid()
    if i==len(analysers)-1:
        ax[i].set_xlabel("slope error (mrad)")
    ax[i].set_ylabel("Frequency (counts per bin)")
    ax[i].set_xlim(0,3.6)
    set_plot_font_size(ax[i],16,False)

In [ ]:
# everything on a single plot (for APSRC abstract)
f,ax = plt.subplots()
f.set_size_inches(14,6)
ax.grid()
colours = ["tab:blue","tab:orange","tab:green"]
ymax = 65
vline_extents = [0.8,0.86,0.9]
annotation_positions = [(1.35,53),(1.2,57),(1.0,61)]
labels = ["Design paraboloid","Fitted paraboloid","Deformed paraboloid"]
for i,analyserName in enumerate(analysers.keys()):
    # ax[i].set_title(analyserName)
    mean = df["sigma (%s)"%analyserName].mean()*1e3
    counts,bins,_=ax.hist(df["sigma (%s)"%analyserName]*1e3,bins=50,rwidth=0.85,alpha=0.5,label=labels[i])
    color = colours[i]
    ax.axvline(mean, color=color, linestyle='dashed', linewidth=3, ymax=vline_extents[i])
    ax.text(annotation_positions[i][0], annotation_positions[i][1], '{:.2f} mrad'.format(mean),color=color,size=18)
ax.set_xlabel("slope error (mrad)")
ax.set_ylabel("Frequency (counts per bin)")
ax.set_ylim(0,ymax)
ax.set_xlim(0.5,2.5)
set_plot_font_size(ax,18,True)
plt.tight_layout()
plt.savefig("field_slope_error_histogram.png")


In [ ]:
# everything on a single plot (for APSRC abstract)
f,ax = plt.subplots()
f.set_size_inches(14,6)
ax.grid()
colours = ["tab:blue","tab:orange","tab:green"]
ymax = 65
vline_extents = [0.8,0.86,0.9]
annotation_positions = [(1.35,53),(1.2,57),(1.0,61)]
labels = ["Design paraboloid","Fitted paraboloid","Deformed paraboloid"]
values = []
for i,analyserName in enumerate(analysers.keys()):
    # ax[i].set_title(analyserName)
    mean = df["sigma (%s)"%analyserName].mean()*1e3
    values.append(df["sigma (%s)"%analyserName]*1e3)
    color = colours[i]
    ax.axvline(mean, color=color, linestyle='dashed', linewidth=3, ymax=vline_extents[i])
    ax.text(annotation_positions[i][0], annotation_positions[i][1], '$\mu$ = {:.2f} mrad'.format(mean),color=color,size=18)
counts,bins,_=ax.hist(values,bins=50,rwidth=0.85,alpha=1.0,label=labels)
ax.set_xlabel("slope error (mrad)")
ax.set_ylabel("Frequency (counts per bin)")
ax.set_ylim(0,ymax)
ax.set_xlim(0.5,2.5)
set_plot_font_size(ax,18,True)
plt.tight_layout()
plt.savefig("field_slope_error_histogram.png")


### plotting of focal length error distribution for the scanned facet population

In [ ]:
f,ax = plt.subplots()
f.set_size_inches(14,6)
ax.grid()
ax.hist(df["eps(f)"]*100,bins=50,rwidth=0.85,alpha=0.5)
ax.set_xlabel("Focal length error (%)")
ax.set_ylabel("Frequency (counts per bin)")
set_plot_font_size(ax,18,False)
plt.tight_layout()
plt.savefig("field_focal_length_error_histogram.png")

###  plotting of slope error spatial distribution for a particular scan

In [ ]:
analyser = analysers["FitSurface (paraboloid)"]

In [ ]:
# useful for finding heliostats
while 0:
    scanFile = getScanFile(type_="random")
    slopeError = df.loc[scanFile.replace(".txt","")]['sigma (FixedFocalLength)']
    if slopeError>0.002:
        print("%s, slope error = %0.3f mrad"%(scanFile,slopeError*1e3))
        scanPath = os.path.join(scansDir,scanFile)
        break

In [ ]:
def getScanFile(type_):
    if type_=="best":
        scanName = df.loc[df['sigma (FixedFocalLength)'].idxmin()].name
    elif type_=="worst":
        scanName = df.loc[df['sigma (FixedFocalLength)'].idxmax()].name
    elif type_=="deformed":
        scanName = "PEG-FA35-BC02-M003"
    elif type_=="mean":
        scanName = abs(df['sigma (FixedFocalLength)']-df['sigma (FixedFocalLength)'].mean()).idxmin()
    elif type_=="mean_custom":
        scanName = "PEG-FA35-BC07-M091"
    elif type_=="random":
        return random.choice(scanFiles)
    else:
        raise RuntimeError("type_: %s not recognised"%type_)
    return scanName+".txt"

In [ ]:
scanFile = getScanFile(type_="deformed")
slopeError = df.loc[scanFile.replace(".txt","")]['sigma (FixedFocalLength)']
print("%s, slope error = %0.3f mrad"%(scanFile,slopeError*1e3))
scanPath = os.path.join(scansDir,scanFile)

### 1. via the analyse_scan function (quick and easy but not very flexible)

In [ ]:
for mesherName,mesher in meshers.items():
    print(mesherName)
    result = AnalysisManager.analyse_scan(scanPath,loader,preprocessor,mesher,analyser,aggregator,output_dir=None,plot=True,write_stls=False)
    f = 1/4./result["coeff: a(x^2 + y^2)"]
    print(result["aggregate"])

### 2. via custom code (allows modification of figure size and data range etc)

In [ ]:
for mesherName,mesher in meshers.items():
    scan_name = os.path.splitext(os.path.basename(scanPath))[0]
    raw_x, raw_y, raw_z = loader.load_points(scanPath)
    x, y, z = preprocessor.preprocess_points(raw_x, raw_y, raw_z)
    mesh = mesher.create_mesh(x, y, z)
    areas, normals, coeffs, ideal_normals, slope_errors = analyser.analyse(x, y, z, mesh, scan_name)
    f,ax = plt.subplots()
    f.set_size_inches(12,10)
    ax.set_title(mesherName)
    plot_2d_surface(mesh,slope_errors,label="slope error (rad)",ax=ax,fontsize=16,cbxy=(4.0,0.29))
    f,ax = plt.subplots()
    f.set_size_inches(14,8)
    ax.set_title(mesherName)
    ax.hist(slope_errors,bins=20,rwidth=0.85)
    ax.grid()
    ax.set_xlabel("slope error (rad)")
    set_plot_font_size(ax,16,False)

### plot slope error contours and slope error histograms for a particular heliostat

In [ ]:
if 1:
    from heliosim_pylib.Heliostat import heliostat
    
    hcDir = os.path.expanduser("~/heliosim-project-files/Heliocon/calibration_target_analyses/Field2/")
    fieldFilePath = os.path.join(hcDir,"HelioControlSystemField_Sparc_August2023.xml")
    seatingPlanFilePath = os.path.join(hcDir,"field2_heliostat_layout.ods")
    
    heliostats = heliostat.getHeliostats(fieldFilePath)
    df_ = pd.read_excel(seatingPlanFilePath,sheet_name="Sheet2")
    df_.set_index("heliostat name",inplace=True)
    
    hid = 190
    hname = None
    for h in heliostats:
        if h.id == hid:
            hname = h.name
            break
    facetName = df.loc[hname]["facet name"]
    found = False
    for scanName_ in scanNames:
        tks = scanName_.split("-")
        facetName_ = "%s-%s-%s"%(tks[0],tks[1],tks[3])
        if facetName_==facetName:
            scanName = scanName_
            found = True
            break
    if not found:
        raise RuntimeError("scanName not found for facetName: %s"%facetName)
    print("hid = %d, hname = %s, scanName = %s"%(hid,hname,scanName))
    
    scanFile = scanName + ".txt"
    scanPath = os.path.join(scansDir,scanFile)
    
    mesher = meshers["Delaunay"]
    scan_name = os.path.splitext(os.path.basename(scanPath))[0]
    raw_x, raw_y, raw_z = loader.load_points(scanPath)
    x, y, z = preprocessor.preprocess_points(raw_x, raw_y, raw_z)
    mesh = mesher.create_mesh(x, y, z)
    areas, normals, coeffs, ideal_normals, slope_errors = analyser.analyse(x, y, z, mesh, scan_name)
    f,ax = plt.subplots()
    f.set_size_inches(12,10)
    ax.set_title(hname)
    plot_2d_surface(mesh,slope_errors,label="slope error (rad)",ax=ax,fontsize=16,cbxy=(4.0,0.29))
    f,ax = plt.subplots()
    f.set_size_inches(14,8)
    ax.set_title(hname)
    ax.hist(slope_errors,bins=20,rwidth=0.85)
    ax.grid()
    ax.set_xlabel("slope error (rad)")
    set_plot_font_size(ax,16,False)

### check sensitivity to aspect ratio

In [ ]:
ax2_parameter = "area"
raw_x, raw_y, raw_z = loader.load_points(scanPath)
x, y, z = preprocessor.preprocess_points(raw_x, raw_y, raw_z)

In [ ]:
max_aspect_ratios = list(np.arange(2,10,1)) + list(np.arange(10,100,10)) + list(np.arange(100,1000,100))
aggregated_slope_errors = []
triangle_counts = []
mirror_areas = []
for max_aspect_ratio in max_aspect_ratios:
    mesher = DelaunayMesher({"max_aspect_ratio":max_aspect_ratio})
    mesh = mesher.create_mesh(x, y, z)
    areas, normals, coeffs, ideal_normals, slope_errors = analyser.analyse(x, y, z, mesh, scan_name)
    triangle_counts.append(len(areas))
    mirror_areas.append(sum(areas))
    aggregate = aggregator.aggregate(slope_errors,areas)
    aggregated_slope_errors.append(aggregate)
f,ax = plt.subplots()
f.set_size_inches(10,8)
ax.plot(max_aspect_ratios,aggregated_slope_errors,color="blue")
ax.set_xlabel("max. aspect ratio")
ax.set_ylabel("aggregated slope error (rad)")
ax.yaxis.label.set_color('blue')
ax.tick_params(axis='y', colors='blue')
ax2 = plt.twinx(ax)
if ax2_parameter=="counts":
    ax2.plot(max_aspect_ratios,triangle_counts,color="r")
    ax2.set_ylabel("triangle counts")
elif ax2_parameter=="area":
    ax2.plot(max_aspect_ratios,mirror_areas,color="r")
    ax2.set_ylabel("mirror area (m$^2$)")
else:
    raise RuntimeError("ax2_parameter: %s not recognised"%ax2_parameter)
ax2.yaxis.label.set_color('red')
ax2.tick_params(axis='y', colors='red')
set_plot_font_size(ax,16,False)
set_plot_font_size(ax2,16,False)
ax.grid()

In [ ]:
max_aspect_ratios = list(np.arange(2,10,1)) + list(np.arange(10,100,10))
aggregated_slope_errors = []
triangle_counts = []
mirror_areas = []
for max_aspect_ratio in max_aspect_ratios:
    mesher = DelaunayMesher({"max_aspect_ratio":max_aspect_ratio})
    mesh = mesher.create_mesh(x, y, z)
    areas, normals, coeffs, ideal_normals, slope_errors = analyser.analyse(x, y, z, mesh, scan_name)
    triangle_counts.append(len(areas))
    mirror_areas.append(sum(areas))
    aggregate = aggregator.aggregate(slope_errors,areas)
    aggregated_slope_errors.append(aggregate)
f,ax = plt.subplots()
f.set_size_inches(10,8)
ax.plot(max_aspect_ratios,aggregated_slope_errors,color="blue")
ax.set_xlabel("max. aspect ratio")
ax.set_ylabel("aggregated slope error (rad)")
ax.yaxis.label.set_color('blue')
ax.tick_params(axis='y', colors='blue')
ax2 = plt.twinx(ax)
if ax2_parameter=="counts":
    ax2.plot(max_aspect_ratios,triangle_counts,color="r")
    ax2.set_ylabel("triangle counts")
elif ax2_parameter=="area":
    ax2.plot(max_aspect_ratios,mirror_areas,color="r")
    ax2.set_ylabel("mirror area (m$^2$)")
else:
    raise RuntimeError("ax2_parameter: %s not recognised"%ax2_parameter)
ax2.yaxis.label.set_color('red')
ax2.tick_params(axis='y', colors='red')
set_plot_font_size(ax,16,False)
set_plot_font_size(ax2,16,False)
ax.grid()

In [ ]:
max_aspect_ratios = list(np.arange(2,10,1))
aggregated_slope_errors = []
triangle_counts = []
mirror_areas = []
for max_aspect_ratio in max_aspect_ratios:
    mesher = DelaunayMesher({"max_aspect_ratio":max_aspect_ratio})
    mesh = mesher.create_mesh(x, y, z)
    areas, normals, coeffs, ideal_normals, slope_errors = analyser.analyse(x, y, z, mesh, scan_name)
    triangle_counts.append(len(areas))
    mirror_areas.append(sum(areas))
    aggregate = aggregator.aggregate(slope_errors,areas)
    aggregated_slope_errors.append(aggregate)
f,ax = plt.subplots()
f.set_size_inches(10,8)
ax.plot(max_aspect_ratios,aggregated_slope_errors,color="blue")
ax.set_xlabel("max. aspect ratio")
ax.set_ylabel("aggregated slope error (rad)")
ax.yaxis.label.set_color('blue')
ax.tick_params(axis='y', colors='blue')
ax2 = plt.twinx(ax)
if ax2_parameter=="counts":
    ax2.plot(max_aspect_ratios,triangle_counts,color="r")
    ax2.set_ylabel("triangle counts")
elif ax2_parameter=="area":
    ax2.plot(max_aspect_ratios,mirror_areas,color="r")
    ax2.set_ylabel("mirror area (m$^2$)")
else:
    raise RuntimeError("ax2_parameter: %s not recognised"%ax2_parameter)
ax2.yaxis.label.set_color('red')
ax2.tick_params(axis='y', colors='red')
set_plot_font_size(ax,16,False)
set_plot_font_size(ax2,16,False)
ax.grid()

In [ ]:
max_aspect_ratios = list(np.linspace(1.001,2,50))
aggregated_slope_errors = []
triangle_counts = []
mirror_areas = []
for max_aspect_ratio in max_aspect_ratios:
    mesher = DelaunayMesher({"max_aspect_ratio":max_aspect_ratio})
    mesh = mesher.create_mesh(x, y, z)
    areas, normals, coeffs, ideal_normals, slope_errors = analyser.analyse(x, y, z, mesh, scan_name)
    triangle_counts.append(len(areas))
    mirror_areas.append(sum(areas))
    aggregate = aggregator.aggregate(slope_errors,areas)
    aggregated_slope_errors.append(aggregate)
f,ax = plt.subplots()
f.set_size_inches(10,8)
ax.plot(max_aspect_ratios,aggregated_slope_errors,color="blue")
ax.set_xlabel("max. aspect ratio")
ax.set_ylabel("aggregated slope error (rad)")
ax.yaxis.label.set_color('blue')
ax.tick_params(axis='y', colors='blue')
ax2 = plt.twinx(ax)
if ax2_parameter=="counts":
    ax2.plot(max_aspect_ratios,triangle_counts,color="r")
    ax2.set_ylabel("triangle counts")
elif ax2_parameter=="area":
    ax2.plot(max_aspect_ratios,mirror_areas,color="r")
    ax2.set_ylabel("mirror area (m$^2$)")
else:
    raise RuntimeError("ax2_parameter: %s not recognised"%ax2_parameter)
ax2.yaxis.label.set_color('red')
ax2.tick_params(axis='y', colors='red')
set_plot_font_size(ax,16,False)
set_plot_font_size(ax2,16,False)
ax.grid()

In [ ]:
max_aspect_ratios = list(np.linspace(1.001,2,50)) + list(np.linspace(2.001,10,50)) + list(np.linspace(10.001,100,50))
aggregated_slope_errors = []
triangle_counts = []
mirror_areas = []
for max_aspect_ratio in max_aspect_ratios:
    mesher = DelaunayMesher({"max_aspect_ratio":max_aspect_ratio})
    mesh = mesher.create_mesh(x, y, z)
    areas, normals, coeffs, ideal_normals, slope_errors = analyser.analyse(x, y, z, mesh, scan_name)
    triangle_counts.append(len(areas))
    mirror_areas.append(sum(areas))
    aggregate = aggregator.aggregate(slope_errors,areas)
    aggregated_slope_errors.append(aggregate)
f,ax = plt.subplots()
f.set_size_inches(10,8)
ax.plot(max_aspect_ratios,aggregated_slope_errors,color="blue")
ax.set_xlabel("max. aspect ratio")
ax.set_ylabel("aggregated slope error (rad)")
ax.yaxis.label.set_color('blue')
ax.tick_params(axis='y', colors='blue')
ax2 = plt.twinx(ax)
if ax2_parameter=="counts":
    ax2.plot(max_aspect_ratios,triangle_counts,color="r")
    ax2.set_ylabel("triangle counts")
elif ax2_parameter=="area":
    ax2.plot(max_aspect_ratios,mirror_areas,color="r")
    ax2.set_ylabel("mirror area (m$^2$)")
else:
    raise RuntimeError("ax2_parameter: %s not recognised"%ax2_parameter)
ax2.yaxis.label.set_color('red')
ax2.tick_params(axis='y', colors='red')
set_plot_font_size(ax,16,False)
set_plot_font_size(ax2,16,False)
ax.grid()